In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_score, recall_score, f1_score)

ROOT = Path(r"C:\Users\Robin Yadav\oulad-early-warning")
PROCESSED = ROOT / "data" / "processed"
CUTOFF = 28

df = pd.read_csv(PROCESSED / f"features_day{CUTOFF}.csv")

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df, df["at_risk"], groups=df["id_student"]))
train, test = df.iloc[train_idx], df.iloc[test_idx]

print("train rows:", len(train), " test rows:", len(test))
print("students in both train and test:",
      len(set(train["id_student"]) & set(test["id_student"])))
print("at-risk rate  train:", round(train["at_risk"].mean(), 3),
      " test:", round(test["at_risk"].mean(), 3))

train rows: 21962  test rows: 5576
students in both train and test: 0
at-risk rate  train: 0.438  test: 0.455


In [9]:
demo_cat = ["code_module", "gender", "region", "highest_education",
            "imd_band", "age_band", "disability"]
demo_num = ["num_of_prev_attempts", "studied_credits", "date_registration"]

behav_num = ["total_clicks", "active_days", "n_sites", "clicks_pre",
             "clicks_days0_13", "clicks_days14_28", "click_trend",
             "days_since_last_active", "no_clicks_flag",
             "n_submitted", "mean_score", "n_assessments_due"]

# deliberately NOT used: id_student (an identifier) and code_presentation
# (a new real-world intake would be an unseen value)

In [7]:
def make_pipe(cat, num, model):
    pre = ColumnTransformer([
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median", add_indicator=True)),
            ("sc", StandardScaler())]), num),
    ])
    return Pipeline([("pre", pre), ("model", model)])

def evaluate(name, pipe, cat, num):
    cols = cat + num
    pipe.fit(train[cols], train["at_risk"])
    proba = pipe.predict_proba(test[cols])[:, 1]
    pred = (proba >= 0.5).astype(int)
    yt = test["at_risk"]
    return {"model": name,
            "roc_auc": roc_auc_score(yt, proba),
            "pr_auc": average_precision_score(yt, proba),
            "precision": precision_score(yt, pred, zero_division=0),
            "recall": recall_score(yt, pred),
            "f1": f1_score(yt, pred)}

In [4]:
results = []
results.append(evaluate("majority-class dummy",
               make_pipe(demo_cat, demo_num, DummyClassifier(strategy="prior")),
               demo_cat, demo_num))
results.append(evaluate("logreg: demographics only",
               make_pipe(demo_cat, demo_num, LogisticRegression(max_iter=2000)),
               demo_cat, demo_num))
results.append(evaluate("logreg: demographics + behavior",
               make_pipe(demo_cat, demo_num + behav_num, LogisticRegression(max_iter=2000)),
               demo_cat, demo_num + behav_num))

pd.DataFrame(results).round(3)

,model,roc_auc,pr_auc,precision,recall,f1
0,majority-class dummy,0.500,0.455,0.000,0.000,0.000
1,logreg: demographics only,0.660,0.614,0.617,0.436,0.511
2,logreg: demographics + behavior,0.784,0.771,0.736,0.586,0.652


In [10]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

def make_pipe_dense(cat, num, model):
    pre = ColumnTransformer([
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat),
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median", add_indicator=True)),
            ("sc", StandardScaler())]), num),
    ], sparse_threshold=0)
    return Pipeline([("pre", pre), ("model", model)])

cat_all = demo_cat
num_all = demo_num + behav_num

In [11]:
results.append(evaluate("random forest",
    make_pipe_dense(cat_all, num_all,
        RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                               n_jobs=-1, random_state=42)),
    cat_all, num_all))

results.append(evaluate("gradient boosting",
    make_pipe_dense(cat_all, num_all,
        HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                       max_leaf_nodes=15, l2_regularization=1.0,
                                       early_stopping=True, random_state=42)),
    cat_all, num_all))

pd.DataFrame(results).round(3)

,model,roc_auc,pr_auc,precision,recall,f1
0,majority-class dummy,0.500,0.455,0.000,0.000,0.000
1,logreg: demographics only,0.660,0.614,0.617,0.436,0.511
2,logreg: demographics + behavior,0.784,0.771,0.736,0.586,0.652
3,random forest,0.788,0.774,0.742,0.604,0.666
4,gradient boosting,0.793,0.785,0.745,0.605,0.668


In [12]:
from sklearn.model_selection import GroupKFold, cross_val_predict

cols = cat_all + num_all
gb = make_pipe_dense(cat_all, num_all,
        HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                       max_leaf_nodes=15, l2_regularization=1.0,
                                       early_stopping=True, random_state=42))

oof = cross_val_predict(gb, train[cols], train["at_risk"],
                        groups=train["id_student"],
                        cv=GroupKFold(n_splits=5),
                        method="predict_proba")[:, 1]
print("done", oof.shape)

done (21962,)


In [13]:
y = train["at_risk"].values
rows = []
for t in np.arange(0.20, 0.75, 0.05):
    pred = (oof >= t).astype(int)
    rows.append({"threshold": round(t, 2),
                 "precision": precision_score(y, pred),
                 "recall": recall_score(y, pred),
                 "f1": f1_score(y, pred),
                 "share_flagged": pred.mean()})
pd.DataFrame(rows).round(3)

,threshold,precision,recall,f1,share_flagged
0,0.20,0.496,0.948,0.651,0.837
1,0.25,0.532,0.900,0.669,0.740
2,0.30,0.571,0.847,0.682,0.650
3,0.35,0.609,0.782,0.684,0.562
4,0.40,0.648,0.711,0.678,0.481
5,0.45,0.685,0.643,0.663,0.411
6,0.50,0.721,0.579,0.642,0.351
7,0.55,0.756,0.512,0.611,0.297
8,0.60,0.790,0.450,0.574,0.249
9,0.65,0.827,0.386,0.526,0.204


In [17]:
from sklearn.base import clone
from sklearn.metrics import confusion_matrix

THRESHOLD = 0.35

final = clone(gb)
final.fit(train[cols], train["at_risk"])
proba_test = final.predict_proba(test[cols])[:, 1]
yt = test["at_risk"].values

print("ROC-AUC:", round(roc_auc_score(yt, proba_test), 3),
      " PR-AUC:", round(average_precision_score(yt, proba_test), 3))

for t in [0.50, THRESHOLD]:
    pred = (proba_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, pred).ravel()
    print(f"\nthreshold {t}: precision={precision_score(yt, pred):.3f} "
          f"recall={recall_score(yt, pred):.3f} f1={f1_score(yt, pred):.3f} "
          f"flagged={pred.mean():.3f}")
    print(f"  caught at-risk (TP)={tp}  missed (FN)={fn}  false alarms (FP)={fp}  correct not-at-risk (TN)={tn}")

ROC-AUC: 0.793  PR-AUC: 0.785

threshold 0.5: precision=0.745 recall=0.605 f1=0.668 flagged=0.370
  caught at-risk (TP)=1535  missed (FN)=1003  false alarms (FP)=526  correct not-at-risk (TN)=2512

threshold 0.35: precision=0.636 recall=0.792 f1=0.705 flagged=0.567
  caught at-risk (TP)=2010  missed (FN)=528  false alarms (FP)=1151  correct not-at-risk (TN)=1887


In [18]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(final, test[cols], test["at_risk"],
                             scoring="roc_auc", n_repeats=5,
                             random_state=42, n_jobs=1)
(pd.Series(imp.importances_mean, index=cols)
   .sort_values(ascending=False).head(10).round(4))

mean_score                0.1022
code_module               0.0387
n_assessments_due         0.0299
highest_education         0.0152
active_days               0.0118
days_since_last_active    0.0087
imd_band                  0.0056
clicks_days0_13           0.0034
num_of_prev_attempts      0.0026
studied_credits           0.0015
dtype: float64

In [19]:
import joblib, json, sklearn

deploy = clone(gb)
deploy.fit(df[cols], df["at_risk"])

MODELS = ROOT / "models"
MODELS.mkdir(exist_ok=True)
joblib.dump(deploy, MODELS / "risk_model_day28.joblib")

meta = {
    "cutoff_day": CUTOFF,
    "threshold": THRESHOLD,
    "cat_features": cat_all,
    "num_features": num_all,
    "sklearn_version": sklearn.__version__,
    "test_roc_auc": float(roc_auc_score(yt, proba_test)),
    "test_pr_auc": float(average_precision_score(yt, proba_test)),
}
(MODELS / "model_meta.json").write_text(json.dumps(meta, indent=2))
print("saved:", [p.name for p in MODELS.iterdir()])

saved: ['model_meta.json', 'risk_model_day28.joblib']


In [16]:
print([n for n in ["train", "test", "gb", "cols", "final", "clone"] if n in globals()])

['train', 'test', 'gb', 'cols']
